# Lezione 10 — Pratica: Dash, callback e interattività

Due esercizi per esercitarti sul pattern `@app.callback` sui due dataset che già conosci: compagnie aeree e incendi in Australia.

**Ricorda il limite tecnico:** Dash non funziona nativamente su Colab. Per testare la tua soluzione:

- **Opzione A:** copia il codice in un file `.py` (puoi usare `%%writefile`) ed eseguilo in un terminale locale con `python file.py`.
- **Opzione B:** cambia l'ultima riga in `app.run(jupyter_mode='inline')` per vederlo incorporato nel notebook (Dash 2.11+ lo include, senza installare nulla di aggiuntivo).

Entrambe le opzioni sono spiegate con codice completo nel notebook di teoria (`clase10_dash-interactividad_IT.ipynb`).

In [ ]:
import pandas as pd

# Dataset delle compagnie aeree (lo userai nell'Esercizio 1)
airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

# Dataset degli incendi in Australia (lo userai nell'Esercizio 2)
wildfire_df = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Historical_Wildfires.csv')
wildfire_df['Month'] = pd.to_datetime(wildfire_df['Date']).dt.month_name()
wildfire_df['Year'] = pd.to_datetime(wildfire_df['Date']).dt.year

airline_data.head()


## Esercizio 1 — Voli per stato di destinazione e compagnia aerea

Costruisci una dashboard con lo stesso pattern dell'**Esempio 1** della lezione (un campo `dcc.Input` per l'anno, un `dcc.Graph` che si ricalcola con un callback), ma questa volta mostrando un **bar chart del numero totale di voli verso ogni stato di destinazione, separato per compagnia aerea (`Reporting_Airline`)**, per l'anno inserito.

**Suggerimenti:**

- Filtra `airline_data` per l'anno come nell'Esempio 1: `df = airline_data[airline_data['Year'] == int(entered_year)]`.
- Raggruppa per `DestState` e `Reporting_Airline`, e conta il numero di voli. Un modo comodo: `df.groupby(['DestState', 'Reporting_Airline'])['Flights'].sum().reset_index()` (la colonna `Flights` in questo dataset indica il numero di voli di ogni riga; potresti anche usare `.size()` se preferisci contare le righe).
- Usa `px.bar(..., x='DestState', y='Flights', color='Reporting_Airline', title=...)`.
- L'Output del callback resta `component_property='figure'` su un `dcc.Graph(id='bar-plot')` vuoto nel layout — come `line-plot` nell'Esempio 1.
- Il titolo dell'app può essere `"Total number of flights to the destination state split by reporting airline"`.


In [ ]:
%%writefile clase10_practica_ejercicio1.py
# TODO: Esercizio 1 - Voli per stato di destinazione e compagnia aerea
import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px

airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

app = dash.Dash(__name__)

# TODO: definisci app.layout
# - Titolo con html.H1 (usa il testo suggerito nei suggerimenti)
# - Un html.Div con "Input Year: " e un dcc.Input(id='input-year', value='2010', type='number')
# - Un dcc.Graph(id='bar-plot') vuoto (senza figure=...)
app.layout = html.Div(children=[
    # ... il tuo codice qui ...
])


# TODO: aggiungi il decoratore @app.callback
# - Output: component_id='bar-plot', component_property='figure'
# - Input: component_id='input-year', component_property='value'
def get_graph(entered_year):
    # TODO:
    # 1. Filtra airline_data per l'anno ricevuto (convertilo in int)
    # 2. Raggruppa per DestState e Reporting_Airline, sommando/contando Flights
    # 3. Crea il bar chart con px.bar (x='DestState', color='Reporting_Airline')
    # 4. Restituisci la figura
    pass


if __name__ == '__main__':
    app.run(debug=True)


<details>
<summary>💡 Vedi soluzione</summary>

```python
# Esercizio 1 - Voli per stato di destinazione e compagnia aerea (soluzione completa)
import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px

# Leggiamo il dataset delle compagnie aeree
airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

app = dash.Dash(__name__)

# Layout: titolo, input dell'anno, e un dcc.Graph vuoto (la figura la riempie il callback)
app.layout = html.Div(children=[
    html.H1('Total number of flights to the destination state split by reporting airline',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 30}),
    html.Div(["Input Year: ", dcc.Input(id='input-year', value='2010',
              type='number', style={'height': '50px', 'font-size': 35}), ],
              style={'font-size': 40}),
    html.Br(),
    html.Br(),
    html.Div(dcc.Graph(id='bar-plot')),
])


@app.callback(Output(component_id='bar-plot', component_property='figure'),
              Input(component_id='input-year', component_property='value'))
def get_graph(entered_year):
    # 1. Filtriamo per l'anno inserito
    df = airline_data[airline_data['Year'] == int(entered_year)]

    # 2. Raggruppiamo per stato di destinazione e compagnia aerea, sommando il numero di voli
    bar_data = df.groupby(['DestState', 'Reporting_Airline'])['Flights'].sum().reset_index()

    # 3. Bar chart, colorato per compagnia aerea
    fig = px.bar(bar_data, x='DestState', y='Flights', color='Reporting_Airline',
                 title='Flights to Destination State, split by Reporting Airline')
    return fig


# Run the app
if __name__ == '__main__':
    app.run(debug=True)
```
</details>


## Esercizio 2 — Un terzo controllo: filtrare la dashboard degli incendi per mese

Prendi la dashboard degli incendi in Australia (Esempio 3 della lezione di teoria) e aggiungi un **terzo controllo**: un `dcc.Dropdown` per scegliere il **mese**, con le opzioni estratte da `wildfire_df.Month.unique()`.

**Suggerimenti:**

1. Aggiungi il nuovo controllo al layout, con il proprio `id` (ad esempio `id='month'`).
2. Aggiungi un terzo `Input(component_id='month', component_property='value')` alla lista di `Input` del callback.
3. Aggiungi `input_month` come terzo argomento della funzione `reg_year_display`, rispettando lo stesso ordine in cui hai dichiarato i tre `Input` (region, year, month).
4. Usa `input_month` per filtrare `y_r_data` di un passo ulteriore (tieni solo le righe di quel mese) **prima** di calcolare `est_data` e `veg_data`.
5. Esegui l'applicazione e conferma che, cambiando uno qualsiasi dei tre controlli — regione, anno o mese — entrambi i grafici (pie e bar) si ricalcolano da soli.


In [ ]:
%%writefile clase10_practica_ejercicio2.py
# TODO: Esercizio 2 - aggiungere un Dropdown di mese alla dashboard degli incendi
import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px

df = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Historical_Wildfires.csv')
df['Month'] = pd.to_datetime(df['Date']).dt.month_name()
df['Year'] = pd.to_datetime(df['Date']).dt.year

app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Australia Wildfire Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 26}),

    html.Div([
        html.Div([
            html.H2('Select Region:', style={'margin-right': '2em'}),
            dcc.RadioItems([{"label": "New South Wales", "value": "NSW"},
                            {"label": "Northern Territory", "value": "NT"},
                            {"label": "Queensland", "value": "QL"},
                            {"label": "South Australia", "value": "SA"},
                            {"label": "Tasmania", "value": "TA"},
                            {"label": "Victoria", "value": "VI"},
                            {"label": "Western Australia", "value": "WA"}],
                           "NSW", id='region', inline=True)]),

        html.Div([
            html.H2('Select Year:', style={'margin-right': '2em'}),
            dcc.Dropdown(df.Year.unique(), value=2005, id='year')
        ]),

        # TODO: aggiungi qui un terzo html.Div con un html.H2('Select Month:', ...)
        # e un dcc.Dropdown(df.Month.unique(), value='January', id='month')

        html.Div([
            html.Div([], id='plot1'),
            html.Div([], id='plot2')
        ], style={'display': 'flex'}),
    ])
])


# TODO: aggiungi 'month' come terzo Input del callback
@app.callback([Output(component_id='plot1', component_property='children'),
               Output(component_id='plot2', component_property='children')],
              [Input(component_id='region', component_property='value'),
               Input(component_id='year', component_property='value')])
def reg_year_display(input_region, input_year):  # TODO: aggiungi input_month come terzo argomento
    region_data = df[df['Region'] == input_region]
    y_r_data = region_data[region_data['Year'] == input_year]

    # TODO: filtra y_r_data anche per input_month prima di continuare

    est_data = y_r_data.groupby('Month')['Estimated_fire_area'].mean().reset_index()
    fig1 = px.pie(est_data, values='Estimated_fire_area', names='Month',
                  title="{} : Monthly Average Estimated Fire Area in year {}".format(input_region, input_year))

    veg_data = y_r_data.groupby('Month')['Count'].mean().reset_index()
    fig2 = px.bar(veg_data, x='Month', y='Count',
                  title='{} : Average Count of Pixels for Presumed Vegetation Fires in year {}'.format(input_region, input_year))

    return [dcc.Graph(figure=fig1),
            dcc.Graph(figure=fig2)]


if __name__ == '__main__':
    app.run(debug=True)


<details>
<summary>💡 Vedi soluzione</summary>

```python
# Esercizio 2 - Dashboard degli incendi con Region + Year + Month (soluzione completa)
import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px

# Leggiamo il dataset degli incendi in Australia
df = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Historical_Wildfires.csv')
df['Month'] = pd.to_datetime(df['Date']).dt.month_name()
df['Year'] = pd.to_datetime(df['Date']).dt.year

app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Australia Wildfire Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 26}),

    html.Div([
        html.Div([
            html.H2('Select Region:', style={'margin-right': '2em'}),
            dcc.RadioItems([{"label": "New South Wales", "value": "NSW"},
                            {"label": "Northern Territory", "value": "NT"},
                            {"label": "Queensland", "value": "QL"},
                            {"label": "South Australia", "value": "SA"},
                            {"label": "Tasmania", "value": "TA"},
                            {"label": "Victoria", "value": "VI"},
                            {"label": "Western Australia", "value": "WA"}],
                           "NSW", id='region', inline=True)]),

        html.Div([
            html.H2('Select Year:', style={'margin-right': '2em'}),
            dcc.Dropdown(df.Year.unique(), value=2005, id='year')
        ]),

        # Nuovo controllo: Dropdown del mese
        html.Div([
            html.H2('Select Month:', style={'margin-right': '2em'}),
            dcc.Dropdown(df.Month.unique(), value='January', id='month')
        ]),

        html.Div([
            html.Div([], id='plot1'),
            html.Div([], id='plot2')
        ], style={'display': 'flex'}),
    ])
])


# Terzo Input aggiunto alla lista, nello stesso ordine in cui viene usato nella funzione
@app.callback([Output(component_id='plot1', component_property='children'),
               Output(component_id='plot2', component_property='children')],
              [Input(component_id='region', component_property='value'),
               Input(component_id='year', component_property='value'),
               Input(component_id='month', component_property='value')])
def reg_year_display(input_region, input_year, input_month):
    # data
    region_data = df[df['Region'] == input_region]
    y_r_data = region_data[region_data['Year'] == input_year]

    # Filtro aggiuntivo per mese, prima di calcolare gli aggregati
    y_r_data = y_r_data[y_r_data['Month'] == input_month]

    # Plot one - Monthly Average Estimated Fire Area
    est_data = y_r_data.groupby('Month')['Estimated_fire_area'].mean().reset_index()
    fig1 = px.pie(est_data, values='Estimated_fire_area', names='Month',
                  title="{} : Estimated Fire Area in {} {}".format(input_region, input_month, input_year))

    # Plot two - Average Count of Pixels for Presumed Vegetation Fires
    veg_data = y_r_data.groupby('Month')['Count'].mean().reset_index()
    fig2 = px.bar(veg_data, x='Month', y='Count',
                  title='{} : Average Count of Pixels for Presumed Vegetation Fires in {} {}'.format(
                      input_region, input_month, input_year))

    return [dcc.Graph(figure=fig1),
            dcc.Graph(figure=fig2)]


if __name__ == '__main__':
    app.run(debug=True)
```

Al cambiare di regione, anno o mese, i tre `Input` attivano lo stesso callback, che restituisce sempre i due grafici già filtrati dalle tre condizioni.
</details>
